<a href="https://colab.research.google.com/github/Antonioufrrj/especializa-o-estatistica/blob/main/trabalho_final_especializa%C3%A7%C3%A3o.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# pandas: Biblioteca amplamente utilizada para manipulação de dados em formato tabular (dataframes).
import pandas as pd
# numpy: Biblioteca que fornece suporte para matrizes multidimensionais e funções matemáticas de alto desempenho.
import numpy as np
# matplotlib.pyplot: Uma das bibliotecas mais populares para criação de gráficos e visualizações de dados em Python.
import matplotlib.pyplot as plt
# matplotlib.dates: Módulo específico do Matplotlib para lidar com datas em gráficos.
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
# seaborn: Biblioteca de visualização de dados baseada no Matplotlib, que oferece uma interface de alto nível para criação de gráficos estatísticos atraentes.
import seaborn as sns
# plotly.express: Biblioteca para criação de gráficos interativos e visualizações de dados.
import plotly.express as px
# requests: Biblioteca para fazer solicitações HTTP, muitas vezes usada para acessar APIs da web.
import requests
# geopandas: Biblioteca para análise e manipulação de dados geoespaciais.
import geopandas as gpd
# datetime: Módulo Python padrão para trabalhar com datas e horários.
import datetime
# openpyxl: Biblioteca para leitura e escrita de arquivos do Microsoft Excel (formato XLSX).
import openpyxl
# os: Módulo Python padrão para interagir com o sistema operacional, incluindo operações de sistema de arquivos.
import os
# calendar: Módulo Python padrão para realizar operações relacionadas a calendários, como obtenção de informações sobre meses e anos.
import calendar
# csv: Módulo Python padrão para leitura e escrita de arquivos CSV.
import csv
import time

import xarray as xr

from shapely.geometry import Point

from sklearn.metrics import mean_squared_error, mean_absolute_error

# Estações INMET

## umidade e temperatura


In [ ]:
path = '/Users/Administrador/Documents/umidade_solo/wsclima/3_custom_relatorio_detalhado.csv' #+ str(num)
pd.read_csv(path)

In [ ]:
path = '/Users/Administrador/Documents/umidade_solo/wsclima' #+ str(num)
caminhos = [os.path.join(path, nome) for nome in os.listdir(path)]
arquivos = [arq for arq in caminhos if os.path.isfile(arq)]
csvs = [arq for arq in arquivos if arq.lower().endswith(".csv")]

df = pd.read_csv(csvs[0])

for i in range(1, len(csvs)):
  data = pd.read_csv(csvs[i])
  df = pd.concat([df, data], axis=0)

cols_to_keep = ['Momento', 'Temperatura Média', 'Umidade Média']
df = df[cols_to_keep]
df['Momento'] = pd.to_datetime(df['Momento'], format='%d/%m/%Y, %H:%M:%S')
df = df.sort_values(by='Momento').reset_index(drop=True)

In [ ]:
display(df.head())

In [ ]:
#df.to_csv('/Users/Administrador/Documents/umidade_solo/wsclima.csv')

In [ ]:
 path = '/Users/Administrador/Documents/umidade_solo/inmet/dados_A706_D_2020-01-01_2025-08-31.csv'
 df_a706 = pd.read_csv(path, sep=',', decimal=',', encoding='latin1')
 df_a706['Data Medicao'] = pd.to_datetime(df_a706['Data Medicao'])

In [ ]:
df_a706

In [ ]:
# Convert 'Momento' in df to just date for merging
df['Momento_date'] = df['Momento'].dt.date

# Convert 'Data Medicao' in df_a706 to datetime and then to date for merging
df_a706['Data Medicao_date'] = pd.to_datetime(df_a706['Data Medicao'], format='%d/%m/%Y').dt.date

# Merge the dataframes on the date columns
merged_df = pd.merge(df_a706, df,  left_on='Data Medicao_date', right_on='Momento_date', how='inner')

# Drop the extra date columns used for merging
merged_df = merged_df.drop(['Momento_date', 'Data Medicao_date'], axis=1)

# Display the head of the merged dataframe
display(merged_df.head())

In [ ]:
#merged_df.to_csv('/Users/Administrador/Documents/umidade_solo/merged_df.csv')

In [ ]:
merged_df[['Temperatura Média',	'TEMPERATURA MEDIA, DIARIA (AUT)(°C)',
           'Umidade Média','UMIDADE RELATIVA DO AR, MEDIA DIARIA (AUT)(%)',
           'PRECIPITACAO TOTAL, DIARIO (AUT)(mm)']].loc[77:].corr()

In [ ]:
df_a706['UMIDADE RELATIVA DO AR, MEDIA DIARIA (AUT)(%)'] = df_a706['UMIDADE RELATIVA DO AR, MEDIA DIARIA (AUT)(%)'].fillna(df['Umidade Média'])
df_a706['TEMPERATURA MEDIA, DIARIA (AUT)(°C)'] = df_a706['TEMPERATURA MEDIA, DIARIA (AUT)(°C)'].fillna(df['Temperatura Média'])


In [ ]:
df_a706.isnull().sum()

## radiação

In [ ]:
nums = [509, 728, 769, 706]
for num in nums:
  nome_variavel = f"df_{num}"

  path = '/Users/Administrador/Documents/umidade_solo/inmet/' + str(num)
  caminhos = [os.path.join(path, nome) for nome in os.listdir(path)]
  arquivos = [arq for arq in caminhos if os.path.isfile(arq)]
  csvs = [arq for arq in arquivos if arq.lower().endswith(".csv")]

  df = pd.read_csv(csvs[0], sep=';', decimal=',', skiprows=8, encoding='latin1')

  for i in range(1, len(csvs)):
    data = pd.read_csv(csvs[i], sep=';', decimal=',', skiprows=8, encoding='latin1')
    df = pd.concat([df, data], axis=0)

  df = df.reset_index(drop=True)
  cols_to_keep = ['Data', 'Hora UTC', 'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
                  'RADIACAO GLOBAL (Kj/m²)']

  df = df[cols_to_keep]
  horas_to_fill_zero = ['2300 UTC', '0000 UTC', '0100 UTC', '0200 UTC', '0300 UTC',
                      '0400 UTC', '0500 UTC', '0600 UTC', '0700 UTC', '0800 UTC']

  df.loc[df['Hora UTC'].isin(horas_to_fill_zero), 'RADIACAO GLOBAL (Kj/m²)'] = 0
  locals()[nome_variavel] = df.copy()

In [ ]:
cols_to_include = ['RADIACAO GLOBAL (Kj/m²)']

nums = [509, 728, 769, 706]
dataframes = {509: df_509, 728: df_728, 769: df_769, 706: df_706}

combined_df = pd.DataFrame()

# Add Data and Hora UTC columns from one of the dataframes (assuming they are the same)
combined_df['Data'] = df_509['Data']
combined_df['Hora UTC'] = df_509['Hora UTC']

# Create a dictionary for renaming columns
rename_dict = {}
for col in cols_to_include:
    for num in nums:
        original_col_name = f'{col}_{num}'
        if 'RADIACAO GLOBAL' in col:
            new_col_name = f'radiacao_{num}'
        else:
            new_col_name = original_col_name # Keep original if no specific rule

        combined_df[original_col_name] = dataframes[num][col]
        rename_dict[original_col_name] = new_col_name

# Rename the columns
combined_df.rename(columns=rename_dict, inplace=True)


display(combined_df.head())

In [ ]:
combined_df.isnull().sum()

In [ ]:
#combined_df.to_csv('/Users/Administrador/Documents/umidade_solo/estacoes.csv', index=False)

In [ ]:
combined_df[['radiacao_509',	'radiacao_728',	'radiacao_769',	'radiacao_706']].corr()

In [ ]:
import statsmodels.api as sm

In [ ]:
import statsmodels.api as sm

# Regression for Radiation
independent_vars_rad = ['radiacao_509']
dependent_var_rad = 'radiacao_706'

df_reg_rad = combined_df.dropna(subset=independent_vars_rad + [dependent_var_rad])

X_rad = df_reg_rad[independent_vars_rad]
y_rad = df_reg_rad[dependent_var_rad]

X_rad = sm.add_constant(X_rad)

model_rad = sm.OLS(y_rad, X_rad)
results_rad = model_rad.fit()

print("\nRadiation Regression Results:")
print(results_rad.summary())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plotting Temperature Regression
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_rad, y=results_rad.predict(X_rad))
plt.xlabel("Observed Temperature")
plt.ylabel("Predicted Temperature")
plt.title("Temperature Regression: Observed vs Predicted")
plt.show()

## teste com regressão

In [ ]:
import statsmodels.api as sm

# Regression for Temperature
independent_vars_temp = ['temperatura_509']
dependent_var_temp = 'temperatura_706'

df_reg_temp = combined_df.dropna(subset=independent_vars_temp + [dependent_var_temp])

X_temp = df_reg_temp[independent_vars_temp]
y_temp = df_reg_temp[dependent_var_temp]

X_temp = sm.add_constant(X_temp)

model_temp = sm.OLS(y_temp, X_temp)
results_temp = model_temp.fit()

print("Temperature Regression Results:")
print(results_temp.summary())

In [ ]:
import statsmodels.api as sm

# Regression for Humidity
independent_vars_hum = ['umidade_509']
dependent_var_hum = 'umidade_706'

df_reg_hum = combined_df.dropna(subset=independent_vars_hum + [dependent_var_hum])

X_hum = df_reg_hum[independent_vars_hum]
y_hum = df_reg_hum[dependent_var_hum]

X_hum = sm.add_constant(X_hum)

model_hum = sm.OLS(y_hum, X_hum)
results_hum = model_hum.fit()

print("\nHumidity Regression Results:")
print(results_hum.summary())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plotting Temperature Regression
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_temp, y=results_temp.predict(X_temp))
plt.xlabel("Observed Temperature")
plt.ylabel("Predicted Temperature")
plt.title("Temperature Regression: Observed vs Predicted")
plt.show()

# Plotting Humidity Regression
plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_hum, y=results_hum.predict(X_hum))
plt.xlabel("Observed Humidity")
plt.ylabel("Predicted Humidity")
plt.title("Humidity Regression: Observed vs Predicted")
plt.show()

In [ ]:
nome_col = ['umidade_706', 'temperatura_706']
for i in nome_col:
  # Identify rows where 'radiacao_706' is missing
  missing_radiacao_706_rows = combined_df[combined_df[i].isnull()].copy()


  # Select the independent variables for the rows with missing 'radiacao_706'
  X_missing = missing_radiacao_706_rows[independent_vars]

  # Add a constant to the independent variables for prediction
  X_missing = sm.add_constant(X_missing)

  # Predict the missing 'radiacao_706' values using the trained model (results)
  predicted_radiacao_706 = results.predict(X_missing)

  # Fill the missing values in the original combined_df DataFrame
  # Use .loc to ensure we are modifying the original DataFrame correctly
  combined_df.loc[missing_radiacao_706_rows.index, i] = predicted_radiacao_706

  # Verify that there are no more missing values in 'radiacao_706' in the filled rows
  # Check null counts for verification
  print("Null values after filling:")
  display(combined_df[i].isnull().sum())

In [ ]:
# Convert 'Data' column to datetime
combined_df['Data'] = pd.to_datetime(combined_df['Data'])

# Define the columns for mean and sum aggregation
temp_hum_cols = [col for col in combined_df.columns if ('temperatura' in col or 'umidade' in col)]
rad_cols = [col for col in combined_df.columns if 'radiacao' in col]

# Group by day and aggregate
daily_df = combined_df.groupby(combined_df['Data'].dt.date).agg(
    {**{col: 'mean' for col in temp_hum_cols},
     **{col: 'sum' for col in rad_cols}}
).reset_index()

# Rename the index column
daily_df.rename(columns={'index': 'Data'}, inplace=True)

display(daily_df.head())

In [ ]:
daily_df[['radiacao_509',	'radiacao_728',	'radiacao_769',	'radiacao_706']].corr()

In [ ]:
horas_to_drop = ['2300 UTC', '0000 UTC', '0100 UTC', '0200 UTC', '0300 UTC',
                 '0400 UTC', '0500 UTC', '0600 UTC', '0700 UTC', '0800 UTC']

# Drop rows where 'Hora UTC' is in the list
df_reg_rad = combined_df[~combined_df['Hora UTC'].isin(horas_to_drop)].reset_index(drop=True)


In [ ]:
import statsmodels.api as sm

# Regression for Radiation
independent_vars_rad = ['radiacao_509']
dependent_var_rad = 'radiacao_706'



df_reg_rad = daily_df.dropna(subset=independent_vars_rad + [dependent_var_rad])

X_rad = df_reg_rad[independent_vars_rad]
y_rad = df_reg_rad[dependent_var_rad]

X_rad = sm.add_constant(X_rad)

model_rad = sm.OLS(y_rad, X_rad)
results_rad = model_rad.fit()

print("\nRadiation Regression Results:")
print(results_rad.summary())

In [ ]:
combined_df

In [ ]:
#df.to_csv('/Users/Administrador/Documents/umidade_solo/estacoes_.csv', index=False)

# tratamento estações

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Definindo uma lista chamada 'caminhos' que conterá os caminhos completos de todos os arquivos na pasta './todos_municipios'.
# O loop 'for' itera sobre cada nome de arquivo na pasta './todos_municipios' e usa a função 'os.path.join()' para criar o caminho completo.
caminhos = [os.path.join('/Users/Administrador/Documents/umidade_solo/todos_municipios', nome) for nome in os.listdir('/Users/Administrador/Documents/umidade_solo/todos_municipios')]

# Definindo uma lista chamada 'arquivos' que conterá apenas os caminhos que correspondem a arquivos (não diretórios).
# Isso é feito filtrando os caminhos da lista 'caminhos' usando a função 'os.path.isfile()'.
arquivos = [arq for arq in caminhos if os.path.isfile(arq)]

# Definindo uma lista chamada 'csvs' que conterá apenas os caminhos que correspondem a arquivos com extensão '.csv'.
# Isso é feito filtrando os caminhos da lista 'arquivos' usando a função 'str.endswith(".csv")' para verificar se a extensão do arquivo é '.csv'.
csvs = [arq for arq in arquivos if arq.lower().endswith(".csv")]

# A lista 'csvs' agora contém os caminhos completos de todos os arquivos CSV na pasta './todos_municipios'.
# Você pode usar essa lista para realizar operações em arquivos CSV específicos.
'''
Este código basicamente cria listas de caminhos para diferentes tipos de arquivos em uma pasta especificada
('./todos_municipios'). Ele começa criando uma lista de todos os caminhos de arquivos na pasta, depois filtra
essa lista para incluir apenas caminhos que correspondem a arquivos (não diretórios) e finalmente filtra ainda
mais para incluir apenas caminhos de arquivos CSV. Os caminhos resultantes estão armazenados na lista 'csvs'
para uso posterior.
'''

In [ ]:
# Lendo o primeiro arquivo CSV da lista 'csvs' e armazenando seus dados em um DataFrame chamado 'df'.
# O parâmetro 'sep' é usado para especificar o separador de campo no arquivo CSV (neste caso, ';').
df = pd.read_csv(csvs[0], sep=';')

# Iniciando um loop 'for' que percorre os outros arquivos CSV na lista 'csvs', começando pelo segundo arquivo.
for i in range(1, len(csvs)):
    # Lendo o arquivo CSV atual da lista 'csvs' e armazenando seus dados em um novo DataFrame chamado 'data'.
    # O parâmetro 'sep' novamente é usado para especificar o separador de campo no arquivo CSV.
    data = pd.read_csv(csvs[i], sep=';')

    # Concatenando o DataFrame 'data' ao DataFrame 'df' ao longo do eixo das linhas (axis=0).
    df = pd.concat([df, data], axis=0)

# Resetando o índice do DataFrame 'df' para que os índices das linhas sejam sequenciais e únicos.
df = df.reset_index(drop=True)

# O DataFrame 'df' agora contém os dados combinados de todos os arquivos CSV da lista 'csvs' com índices redefinidos.
# Você pode usar o DataFrame 'df' para análise e manipulação de dados tabulares consolidados.
'''
Este código lê vários arquivos CSV da lista 'csvs', combina-os em um único DataFrame chamado 'df' e redefine os índices
das linhas para garantir que sejam sequenciais e únicos no DataFrame resultante. Isso é útil quando você deseja realizar
análises ou manipulações em dados de vários arquivos CSV de maneira consolidada.
'''

In [ ]:
# Convertendo a coluna 'datahora' para o tipo de dado datetime.
df['datahora'] = pd.to_datetime(df['datahora'])

# Removendo a informação de fuso horário da coluna 'datahora', definindo-a como nula.
df['datahora'] = df['datahora'].dt.tz_localize(None)
# Removendo as colunas 'codEstacao' e 'uf' do DataFrame 'df'.
# O parâmetro 'axis=1' indica que as colunas devem ser removidas (axis=1 refere-se às colunas, axis=0 refere-se às linhas).
df = df.drop(['codEstacao', 'uf'], axis=1)
# Substituindo vírgulas por pontos na coluna 'valorMedida'.
df['valorMedida'] = df['valorMedida'].str.replace(',', '.')
# Convertendo a coluna 'valorMedida' para o tipo de dado numérico (float).
df['valorMedida'] = pd.to_numeric(df['valorMedida'])
# Usando a função 'isnull()' para verificar se há valores nulos em cada coluna do DataFrame 'df'.
# Em seguida, usando 'sum()' para contar o número de valores nulos em cada coluna.
# Finalmente, imprimindo a contagem de valores nulos em cada coluna.
print(df.isnull().sum())

In [ ]:
# Avaliando a expressão (df['valorMedida'] < 0) que cria uma série booleana indicando se cada valor na coluna 'valorMedida' é menor que zero.
# Em seguida, usando 'sum()' para contar quantos valores True (ou seja, valores menores que zero) existem na série.
# O resultado é a contagem de valores na coluna 'valorMedida' que são menores que zero.
(df['valorMedida'] < 0).sum()
# Avaliando a expressão (df['valorMedida'] < 0) que cria uma série booleana indicando se cada valor na coluna 'valorMedida' é menor que zero.
# Em seguida, usando essa série booleana como índice para o DataFrame 'df'.
# O resultado é uma seleção de linhas do DataFrame onde os valores na coluna 'valorMedida' são menores que zero.
# Finalmente, usando '.index' para obter os índices das linhas que atendem à condição.
df[df['valorMedida'] < 0].index
# Usando a expressão (df['valorMedida'] < 0) para criar uma série booleana indicando se cada valor na coluna 'valorMedida' é menor que zero.
# Em seguida, usando essa série booleana como índice para selecionar as linhas do DataFrame 'df' onde os valores na coluna 'valorMedida' são menores que zero.
# Usando a função 'drop()' para remover as linhas selecionadas com base nos índices obtidos.
# O parâmetro 'axis=0' indica que as linhas devem ser removidas.
df = df.drop(df[df['valorMedida'] < 0].index, axis=0)

In [ ]:
# Salvando o DataFrame 'df' em um arquivo CSV chamado 'todos_dados_municipios.csv'.
# O parâmetro 'index=False' é usado para não incluir o índice das linhas no arquivo CSV.
#df.to_csv('/Users/Administrador/Documents/umidade_solo/todos_dados_municipios.csv', index=False)


In [ ]:
# Lendo o arquivo CSV 'todos_dados_municipios.csv' e armazenando os dados em um DataFrame chamado 'df'.
# O parâmetro 'sep' é usado para especificar o separador de campo no arquivo CSV (neste caso, ',').
df = pd.read_csv('/Users/Administrador/Documents/umidade_solo/todos_dados_municipios.csv', sep=',')

# Convertendo a coluna 'datahora' para o tipo de dado datetime.
df['datahora'] = pd.to_datetime(df['datahora'])

# Removendo a informação de fuso horário da coluna 'datahora', definindo-a como nula.
df['datahora'] = df['datahora'].dt.tz_localize(None)

# Exibindo as primeiras linhas do DataFrame 'df' para visualização.
df.head()

In [ ]:
df['datahora'] = pd.to_datetime(df['datahora']) - pd.Timedelta(hours=3)

In [ ]:
# Obtendo os valores únicos da coluna 'municipio' do DataFrame 'df' e armazenando-os na variável 'municipio'.
municipio = df['municipio'].unique()
municipio

In [ ]:
nomeEstacao = df['nomeEstacao'].unique()
nomeEstacao

In [ ]:
df['latitude'] = df['latitude'].str.replace(',', '.')
df['longitude'] = df['longitude'].str.replace(',', '.')
df['latitude'] = pd.to_numeric(df['latitude'])#.round(2)
df['longitude'] = pd.to_numeric(df['longitude'])#.round(2)

In [ ]:
df_filtrado = df[["nomeEstacao", "latitude", "longitude"]].drop_duplicates()
df_filtrado = df_filtrado.drop_duplicates(subset=["nomeEstacao"], keep="first")

In [ ]:
#df_filtrado.to_csv('/Users/Administrador/Documents/umidade_solo/estacoes_sp.csv', index=False)

In [ ]:
# Calculando o comprimento (número de elementos) da sequência ou lista 'nomeEstacao'.
# O resultado será a quantidade de elementos na lista 'nomeEstacao'.
len(nomeEstacao)

In [ ]:
#Lista chamada nomes_df que contém uma série de nomes de estações
nomes_df = ['cachoeira_paulista', 'frei_orestes',     'vila_albertina', 'pinhal_miranda', 'duarte_caetano',
            'vila_baiana',        'jardim_alice',     'vila_pregresso', 'nicho_monte_serrat',
            'nova_cintra',        'morro_marape',     'sao_bento',      'caminho_monte_serrat',   'parque_tecnologico',
            'jose_menino',        'barbosas',         'itarare',        'voturua',        'albertina_leste',
            'albertina_centro',   'albertina_torre',  'albertina_oeste','vianas',         'riviera',
            'elias_costa',        'vila_magini',      'zaira_viii', 'boa_esperanca', 'feital', 'zaira_vi', 'zaira_deise',
            'ribeirao_pires', 'sao_joao', 'joao_dias', 'rio_grande', 'palmira_grassioto', 'tondi_lima', 'alice_geotec',
            'sabesp_vila_britania', 'xixova_geotec', 'gerassi','jardim_albamar', 'paranapua']

In [ ]:
len(nomes_df)

In [ ]:
sensor = df['sensor'].unique()
sensor

In [ ]:
# Filtrando o DataFrame 'df' para selecionar apenas as linhas onde a coluna 'sensor' é igual ao valor da primeira
# entrada na lista 'sensor'.
# Em seguida, removendo a coluna 'sensor' do DataFrame resultante com 'drop'.
# Por fim, redefinindo os índices do DataFrame resultante para que sejam sequenciais e únicos.

sensor_1_todos_municipios = df[df["sensor"] == sensor[0]].drop('sensor', axis=1).reset_index(drop=True)
sensor_2_todos_municipios = df[df["sensor"] == sensor[1]].drop('sensor', axis=1).reset_index(drop=True)
sensor_3_todos_municipios = df[df["sensor"] == sensor[2]].drop('sensor', axis=1).reset_index(drop=True)
sensor_4_todos_municipios = df[df["sensor"] == sensor[3]].drop('sensor', axis=1).reset_index(drop=True)
sensor_5_todos_municipios = df[df["sensor"] == sensor[4]].drop('sensor', axis=1).reset_index(drop=True)
sensor_6_todos_municipios = df[df["sensor"] == sensor[5]].drop('sensor', axis=1).reset_index(drop=True)
chuva_todos_municipios    = df[df["sensor"] == sensor[6]].drop('sensor', axis=1).reset_index(drop=True)

In [ ]:
print(len(chuva_todos_municipios),
len(sensor_1_todos_municipios),
len(sensor_2_todos_municipios),
len(sensor_3_todos_municipios),
len(sensor_4_todos_municipios),
len(sensor_5_todos_municipios),
len(sensor_6_todos_municipios))

In [ ]:
# Iterar através de cada estação e nome correspondente
for estacao, nome in zip(nomeEstacao, nomes_df):
    # Criar um nome de variável dinâmica com base no nome da estação
    nome_variavel = f"df_{nome}"
    nchuva = f"chuva_{nome}"


    # Criar uma figura com subplots
    fig, ax1 = plt.subplots(figsize=(12, 7))

    # Filtrar os dados do sensor 3 para a estação atual e redefinir os índices
    locals()[nome_variavel] = sensor_3_todos_municipios[sensor_3_todos_municipios['nomeEstacao'] == estacao].reset_index(drop=True)
    s1 = sensor_1_todos_municipios[sensor_1_todos_municipios['nomeEstacao'] == estacao].reset_index(drop=True)
    s2 = sensor_2_todos_municipios[sensor_2_todos_municipios['nomeEstacao'] == estacao].reset_index(drop=True)

    # Imprimir o nome da variável
    print(nome_variavel,
          locals()[nome_variavel]['datahora'].iloc[0],
          locals()[nome_variavel]['datahora'].iloc[-1])


    # Filtrar os dados de chuva para a estação atual e redefinir os índices
    locals()[nchuva] = chuva_todos_municipios[chuva_todos_municipios['nomeEstacao'] == estacao].reset_index(drop=True)

    # Plotar os valores do sensor 3
    ax1.plot(locals()[nome_variavel]['datahora'], locals()[nome_variavel]['valorMedida'], color='blue', label='sensor 3 - 1.5m')
    ax1.plot(s1['datahora'], s1['valorMedida'], color='red', label='sensor 1 - 0.5m')
    ax1.plot(s2['datahora'], s2['valorMedida'], color='yellow', label='sensor 2 - 1.0m')
    # Configurar o eixo y para 'sensor 3'
    ax1.set_ylabel('Valores da Umidade', color='blue', fontsize=12, fontweight='bold')#, fontfamily='Times New Roman')
    ax1.tick_params(axis='y', labelcolor='black')
    ax1.legend(bbox_to_anchor=(1.10, 0.52), loc=2, borderaxespad=0.)#, prop={'family': 'Times New Roman', "weight": 'bold', "size": 12})

    ax1.tick_params(axis='x', rotation=45)

    # Criar um segundo eixo y
    ax2 = ax1.twinx()

    # Plotar os valores de precipitação
    ax2.plot(locals()[nchuva]['datahora'], locals()[nchuva]['valorMedida'], color='black', label='precipitação')

    # Configurar o eixo y para 'chuva'
    ax2.tick_params(axis='y', labelcolor='black')
    ax2.legend(bbox_to_anchor=(1.10, 0.62), loc=2, borderaxespad=0.)#, prop={'family': 'Times New Roman', "weight": 'bold', "size": 12})

    # Configurar título do subplot
    ax1.set_title('todos os anos', fontsize=14, fontweight='bold')#, fontfamily='Times New Roman')

    # Configurar rótulos dos eixos
    ax2.set_ylabel('Valores de Chuva', color='red', fontsize=12, fontweight='bold')#, fontfamily='Times New Roman')
    ax2.set_xlabel('Data', color='black', fontsize=12, fontweight='bold')#, fontfamily='Times New Roman')

    # Configurar estilo de fonte
    #plt.rcParams["font.family"] = "Times New Roman"
    plt.rcParams["font.size"] = "10"
    plt.rcParams["font.weight"] = "bold"

    # Exibir o gráfico
    plt.show()


In [ ]:
df['nomeEstacao'].unique()

In [ ]:
estacao_funcionando = ['Jd. Frei Orestes', 'EE Duarte Caetano - Cota 200', 'Vila Baiana',
    'Nicho VI - Monte Serrat', 'Seminário - Morro Nova Cintra',
    'Morro Itararé', 'Vila Albertina Rua F Centro', 'Vila Albertina Torre Vanguarda',
    'Vila Magini', 'Feital', 'EGEO Ribeirão Pires','Riviera', 'João Dias', 'ETA Rio Grande',
    'Palmira Grassioto', 'Brazilia Tondi de Lima', 'ALICE GEOTEC', 'Xixová Geotec', 'Parque Gerassi']
"""
estacao_nfuncionam = ['frei_orestes', 'duarte_caetano', 'vila_baiana','nicho_monte_serrat',
            'nova_cintra',       'itarare',   'albertina_centro',   'albertina_torre',
            'vila_magini', 'feital','ribeirao_pires', 'riviera', 'joao_dias', 'rio_grande',
            'palmira_grassioto', 'tondi_lima', 'alice_geotec', 'xixova_geotec', 'gerassi']
"""
# Filtra o DataFrame original para incluir apenas as estações que estão enviados dados
df_funcionando = df[df["nomeEstacao"].isin(estacao_funcionando)].reset_index(drop=True)

In [ ]:
estacao_nfuncionam = ['Cachoeira Paulista (TESTES GEO)',  'Vila Albertina', 'Pinhal do Miranda',
  'Jardim Alice', 'EE Emílio Justo - Vila Progresso', 'HUVET - Morro Marapé',
  'Caminho - Monte Serrat', 'Parque Tecnológico SJC (TESTES GEO)', 'Sabesp - Morro São Bento',
  'Sabesp - Morro José Menino', 'Morro Barbosas', 'Morro Voturuá', 'Vila Albertina Rua F Leste',
  'Vila Albertina Rua F Oeste', 'Sítio dos Vianas', 'Elias Costa',
  'Jardim Zaira VIII', 'Parque Boa Esperança', 'Jardim Zaira VI', 'Jardim Zaira - Deise',
  'Vila São João',  'Jardim Albamar', 'Paranapuã', 'Sabesp - Vila Britânia']

"""
estacao_nfuncionam =   ['cachoeira_paulista','vila_albertina',   'pinhal_miranda',
             'vila_pregresso',    'morro_marape',     'caminho_monte_serrat',   'parque_tecnologico',
             'jose_menino',       'barbosas',         'voturua',                'albertina_leste',
             'albertina_oeste',   'vianas',                 'elias_costa', 'sao_bento',
             'zaira_viii',       'boa_esperanca',     'zaira_vi',               'zaira_deise',
             'sao_joao',         'alice_geotec',      'jardim_albamar',         'paranapua']
"""
# Filtra o DataFrame original para incluir apenas as estações que não estão enviados dados
df_nfuncionam = df[df["nomeEstacao"].isin(estacao_nfuncionam)].reset_index(drop=True)

In [ ]:
len(df_funcionando['nomeEstacao'].unique())

In [ ]:
for estacao, nome in zip(df_funcionando['nomeEstacao'].unique(), estacao_funcionando):

    # Criar um nome de variável dinâmica com base no nome da estação
    nome_variavel = f"df_{nome}"
    nchuva = f"chuva_{nome}"


    # Criar uma figura com subplots
    fig, ax1 = plt.subplots(figsize=(12, 7))

    # Filtrar os dados do sensor 3 para a estação atual e redefinir os índices
    locals()[nome_variavel] = sensor_3_todos_municipios[sensor_3_todos_municipios['nomeEstacao'] == estacao].reset_index(drop=True)
    s1 = sensor_1_todos_municipios[sensor_1_todos_municipios['nomeEstacao'] == estacao].reset_index(drop=True)
    s2 = sensor_2_todos_municipios[sensor_2_todos_municipios['nomeEstacao'] == estacao].reset_index(drop=True)

    # Imprimir o nome da variável
    print(nome_variavel,
          locals()[nome_variavel]['datahora'].iloc[0],
          locals()[nome_variavel]['datahora'].iloc[-1])


    # Filtrar os dados de chuva para a estação atual e redefinir os índices
    locals()[nchuva] = chuva_todos_municipios[chuva_todos_municipios['nomeEstacao'] == estacao].reset_index(drop=True)


    # Plotar os valores do sensor 3
    #ax1.plot(locals()[nome_variavel]['datahora'], locals()[nome_variavel]['valorMedida'], color='blue', label='sensor 3 - 1.5m')
    ax1.plot(s1['datahora'], s1['valorMedida'], color='red', label='sensor 1 - 0.5m')
    ax1.plot(s2['datahora'], s2['valorMedida'], color='yellow', label='sensor 2 - 1.0m')
    # Configurar o eixo y para 'sensor 3'
    ax1.set_ylabel('Valores da Umidade', color='blue', fontsize=12, fontweight='bold')#, fontfamily='Times New Roman')
    ax1.tick_params(axis='y', labelcolor='black')
    ax1.legend(bbox_to_anchor=(1.10, 0.52), loc=2, borderaxespad=0.)#, prop={'family': 'Times New Roman', "weight": 'bold', "size": 12})

    ax1.tick_params(axis='x', rotation=45)

    # Criar um segundo eixo y
    ax2 = ax1.twinx()

    # Plotar os valores de precipitação
    ax2.plot(locals()[nchuva]['datahora'], locals()[nchuva]['valorMedida'], color='black', label='precipitação')

    # Configurar o eixo y para 'chuva'
    ax2.tick_params(axis='y', labelcolor='black')
    ax2.legend(bbox_to_anchor=(1.10, 0.62), loc=2, borderaxespad=0.)#, prop={'family': 'Times New Roman', "weight": 'bold', "size": 12})

    # Configurar título do subplot
    ax1.set_title('todos os anos', fontsize=14, fontweight='bold')#, fontfamily='Times New Roman')

    # Configurar rótulos dos eixos
    ax2.set_ylabel('Valores de Chuva', color='red', fontsize=12, fontweight='bold')#, fontfamily='Times New Roman')
    ax2.set_xlabel('Data', color='black', fontsize=12, fontweight='bold')#, fontfamily='Times New Roman')

    # Configurar estilo de fonte
    #plt.rcParams["font.family"] = "Times New Roman"
    plt.rcParams["font.size"] = "10"
    plt.rcParams["font.weight"] = "bold"

    # Exibir o gráfico
    plt.show()


In [ ]:
# Lista de nomes de estações a serem mantidas
estacao_escolhida = ['Jd. Frei Orestes', 'Feital', 'Palmira Grassioto', 'Xixová Geotec']
#Lista chamada nomes_df que contém uma série de nomes de estações
nomes_df = ['frei_orestes', 'feital','palmira_grassioto', 'geotec']

In [ ]:
cols = ['datahora', 'sensor1-50cm',	  'sensor2-100cm',	'sensor3-150cm',
        'sensor4-200cm',	'sensor5-250cm',	'sensor6-300cm', 'precipitação']
for estacao, nome in zip(estacao_escolhida, nomes_df):
  nome_variavel = f"df_{nome}"
  print(nome_variavel)
  locals()[nome_variavel]  = df[df["nomeEstacao"] == estacao].reset_index(drop=True)
  # Use pivot_table para converter a coluna 'sensor' em colunas
  locals()[nome_variavel] = pd.pivot_table(locals()[nome_variavel], values='valorMedida',
                        index=['datahora'],
                        columns='sensor').reset_index()
  locals()[nome_variavel].columns = range(len(locals()[nome_variavel].columns))
  novos_nomes = {
      0 : 'datahora',
      2 : 'sensor1-50cm',
      3 : 'sensor2-100cm',
      4 : 'sensor3-150cm',
      5 : 'sensor4-200cm',
      6 : 'sensor5-250cm',
      7 : 'sensor6-300cm',
      1 : 'precipitação',
  }
  # Renomear as colunas usando o método rename
  locals()[nome_variavel] = locals()[nome_variavel].rename(columns=novos_nomes)
  locals()[nome_variavel] = locals()[nome_variavel][cols]
  locals()[nome_variavel]['precipitação'].fillna(0, inplace=True)
  colss = ['sensor1-50cm',	  'sensor2-100cm',	'sensor3-150cm',
        'sensor4-200cm',	'sensor5-250cm',	'sensor6-300cm']
  Q1 = locals()[nome_variavel][colss].quantile(0.25)
  Q3 = locals()[nome_variavel][colss].quantile(0.75)
  IQR = Q3 - Q1

  lower_limit = Q1 - 1.5 * IQR
  lower_limit
  # Filtre os valores abaixo do limite inferior em todas as colunas
  for col in colss:
      locals()[nome_variavel] = locals()[nome_variavel][locals()[nome_variavel][col] >= lower_limit[col]]
  locals()[nome_variavel] = locals()[nome_variavel].reset_index(drop=True)

  # Data inicial e final do intervalo desejado (substitua pelas suas datas)
  data_inicial = str(locals()[nome_variavel]['datahora'][0])
  data_final = str(locals()[nome_variavel]['datahora'].iloc[-1])

  # Criação do intervalo de datas com frequência de 10 minutos
  intervalo_de_datas = pd.date_range(start=data_inicial, end=data_final, freq='10T')

  # Este código cria um objeto de índice de data e hora que representa todas as datas e horas em um intervalo entre a data inicial e final, com intervalos de 10 minutos.

  # O resultado, 'intervalo_de_datas', pode ser usado para gerar sequências de datas e horas em intervalos específicos, como a cada 10 minutos.
  # Criar um DataFrame com as datas
  intervalo = pd.DataFrame({'datahora': intervalo_de_datas})
  locals()[nome_variavel] = pd.merge(intervalo, locals()[nome_variavel], how = 'left', on = 'datahora')
  locals()[nome_variavel].fillna(method='ffill', inplace=True)  # Preenchimento com valor anterior
  df_acumulado = locals()[nome_variavel].groupby(pd.Grouper(key='datahora', freq='10T'))['precipitação'].sum().reset_index()

  #locals()[nome_variavel].to_csv('/Users/Administrador/Documents/umidade_solo/'+ nome_variavel +'.csv', index=False)

In [ ]:
s1 = sensor_1_todos_municipios[sensor_1_todos_municipios['nomeEstacao'] == 'EGEO Ribeirão Pires'].reset_index(drop=True)
s2 = sensor_2_todos_municipios[sensor_2_todos_municipios['nomeEstacao'] == 'EGEO Ribeirão Pires'].reset_index(drop=True)
chuva = chuva_todos_municipios[chuva_todos_municipios['nomeEstacao'] == 'EGEO Ribeirão Pires'].reset_index(drop=True)


In [ ]:
s1 = s1[s1['valorMedida'] <= 60].reset_index(drop=True)
s2 = s2[s2['valorMedida'] <= 60].reset_index(drop=True)

In [ ]:
fig = plt.figure(figsize=(12, 7))
gs = gridspec.GridSpec(2, 1, height_ratios=[2, 1])  # Define a proporção do espaço vertical (2/3 para o primeiro gráfico e 1/3 para o segundo gráfico)

ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1], sharex=ax1)

# Gráfico de Umidade do Solo
ax1.plot(s1['datahora'], s1['valorMedida'], color='red', label='sensor 1 - 0.5m')
ax1.plot(s2['datahora'], s2['valorMedida'], color='yellow', label='sensor 2 - 1.0m')

ax1.set_ylabel('Umidade do solo (%)', color='black', fontsize=15)
ax1.set_title('Gráfico de Umidade do Solo e Precipitação', fontsize=16, fontweight='bold')
ax1.tick_params(axis='x', rotation=90)
ax1.grid(True)
# Gráfico de Precipitação
ax2.plot(chuva['datahora'], chuva['valorMedida'], color='black', label='precipitação')

ax2.set_ylabel('Precipitação', color='black', fontsize=15)
ax2.tick_params(axis='y', labelcolor='black')
ax2.tick_params(axis='x', rotation=45)
ax2.grid(True)

locator = mdates.MonthLocator(interval=3)  # Define a cada 2 meses
formatter = mdates.DateFormatter('%Y-%m')

ax2.xaxis.set_major_locator(locator)
ax2.xaxis.set_major_formatter(formatter)
# Adicionando as legendas
fig.legend(loc='upper center', bbox_to_anchor=(0.5, -0.005), ncol=4, fontsize=12)

# Ajustando layout
fig.tight_layout()
fig.subplots_adjust(hspace=0.0)  # Reduzir o espaço vertical entre os subplots
plt.show()

# Estações escolhidas

In [ ]:
df_frei_orestes = pd.read_csv('/Users/Administrador/Documents/umidade_solo/df_frei_orestes.csv')
df_frei_orestes['datahora'] = pd.to_datetime(df_frei_orestes['datahora'])
df_frei_orestes.head()

In [ ]:
df_frei_orestes.set_index('datahora', inplace=True)

df_frei_orestes_d = df_frei_orestes.resample('D').mean()
df_frei_orestes_d['precipitação'] = df_frei_orestes['precipitação'].resample('D').sum()
df_frei_orestes_d = df_frei_orestes_d.reset_index()


In [ ]:
fig = plt.figure(figsize=(12, 7))
gs = gridspec.GridSpec(2, 1, height_ratios=[2, 1])  # Define a proporção do espaço vertical (2/3 para o primeiro gráfico e 1/3 para o segundo gráfico)

ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1], sharex=ax1)

# Gráfico de Umidade do Solo
ax1.plot(df_frei_orestes_d['datahora'], df_frei_orestes_d["sensor1-50cm"], color='black', label='0 - 50 cm')
ax1.plot(df_frei_orestes_d['datahora'], df_frei_orestes_d["sensor2-100cm"], color='yellow', label='50 - 100 cm')
ax1.plot(df_frei_orestes_d['datahora'], df_frei_orestes_d["sensor3-150cm"], color='red', label='100 - 150 cm')

ax1.set_ylabel('Umidade do solo (%)', color='black', fontsize=15)
ax1.set_title('Gráfico de Umidade do Solo e Precipitação', fontsize=16, fontweight='bold')
ax1.tick_params(axis='x', rotation=90)
ax1.grid(True)
# Gráfico de Precipitação
ax2.plot(df_frei_orestes_d['datahora'], df_frei_orestes_d['precipitação'], color='blue', alpha=0.5, label='Precipitação (mm)')

ax2.set_ylabel('Precipitação', color='black', fontsize=15)
ax2.tick_params(axis='y', labelcolor='black')
ax2.tick_params(axis='x', rotation=45)
ax2.grid(True)

locator = mdates.MonthLocator(interval=3)  # Define a cada 2 meses
formatter = mdates.DateFormatter('%Y-%m')

ax2.xaxis.set_major_locator(locator)
ax2.xaxis.set_major_formatter(formatter)
# Adicionando as legendas
fig.legend(loc='upper center', bbox_to_anchor=(0.5, -0.005), ncol=4, fontsize=12)

# Ajustando layout
fig.tight_layout()
fig.subplots_adjust(hspace=0.0)  # Reduzir o espaço vertical entre os subplots
plt.show()

In [ ]:
df_frei_orestes_d.head()

In [ ]:
# Crie uma nova coluna para armazenar a precipitação antecedente de 7 dias.
df_frei_orestes_d['precipitacao_3_dias'] = 0.0
df_frei_orestes_d['precipitacao_7_dias'] = 0.0
df_frei_orestes_d['precipitacao_14_dias'] = 0.0
# Defina a janela de 7 dias (7 dias * 24  intervalos de 1 leitura por hora).
janela_7 =  3
janela_30 = 7
janela_60 = 14

df_frei_orestes_d['precipitacao_3_dias'] = df_frei_orestes_d['precipitação'].rolling(window=janela_7, min_periods=1).sum()
df_frei_orestes_d['precipitacao_7_dias'] = df_frei_orestes_d['precipitação'].rolling(window=janela_30, min_periods=1).sum()
df_frei_orestes_d['precipitacao_14_dias'] = df_frei_orestes_d['precipitação'].rolling(window=janela_60, min_periods=1).sum()

In [ ]:
df_frei_orestes_d.rename(columns={'sensor1-50cm': '0 - 50 cm', 'sensor2-100cm': '50 - 100 cm', 'sensor3-150cm': '100 - 150 cm'}, inplace=True)
columns = ['0 - 50 cm', '50 - 100 cm', '100 - 150 cm', 'precipitacao_3_dias', 'precipitacao_7_dias', 'precipitacao_14_dias', 'precipitação']

In [ ]:
'''
fig, ax1 = plt.subplots(figsize=(12,6))

ax1.plot(data_month['datahora'], data_month["sensor1-50cm"], color='black', label='Sensor 1 - 50 cm')
ax1.plot(data_month['datahora'], data_month["sensor2-100cm"], color='yellow', label='Sensor 2 - 100 cm')
ax1.plot(data_month['datahora'], data_month["sensor3-150cm"], color='red', label='Sensor 3 - 150 cm')

ax1.set_ylabel('Valores da Umidade do solo (%)', color='black', fontsize=15)
ax1.tick_params(axis='y', labelcolor='black')
ax1.tick_params(axis='x', rotation=45)
plt.grid()

# Ajustando o eixo principal para mostrar valores entre 30 e 50
#ax1.set_ylim(30, 50)

ax2 = ax1.twinx()
ax2.plot(data_month['datahora'], data_month['precipitação'], color='blue', alpha=0.5, label='Precipitação (mm)')

# Inverter o eixo y da precipitação
#ax2.invert_yaxis()

ax2.tick_params(axis='y', labelcolor='blue')
ax1.set_title('Gráfico de Umidade do Solo e Precipitação', fontsize=16, fontweight='bold')

ax2.set_ylabel('Valores de Precipitação', color='blue', fontsize=15, fontweight='bold')
ax2.set_xlabel('Data', color='black', fontsize=15, fontweight='bold')

fig.legend(loc='upper center', bbox_to_anchor=(0.5, -0.005), ncol=4, fontsize=12)
plt.show()
'''


In [ ]:
df_frei_orestes_d['media'] = (df_frei_orestes_d['0 - 50 cm'] + df_frei_orestes_d['50 - 100 cm'] + df_frei_orestes_d['100 - 150 cm'])/3
#df_frei_orestes_d.to_csv('/Users/Administrador/Documents/umidade_solo/df_frei_orestes_d.csv', index=False)

In [ ]:
df_frei_orestes_d.set_index('datahora', inplace=True)

df_frei_orestes_m = df_frei_orestes_d.resample('M').mean()
df_frei_orestes_m['precipitação'] = df_frei_orestes_d['precipitação'].resample('M').sum()
df_frei_orestes_d = df_frei_orestes_d.reset_index()

In [ ]:
df_frei_orestes_d = df_frei_orestes_d[df_frei_orestes_d['datahora'] >= '2020-01-01'].reset_index(drop=True)


In [ ]:
df_completo = pd.merge(df_frei_orestes_d, df_a706,  left_on='datahora', right_on='Data Medicao', how='inner')

In [ ]:
#df_frei_orestes_d.to_csv('/Users/Administrador/Documents/umidade_solo/df_frei_orestes_h.csv', index=False)

# analise serie temporal

In [ ]:
df_completo.rename(columns={
    'TEMPERATURA MEDIA, DIARIA (AUT)(°C)': 'temperatura',
    'UMIDADE RELATIVA DO AR, MEDIA DIARIA (AUT)(%)': 'umidade_ar'
}, inplace=True)

In [ ]:
columns = ['0 - 50 cm', '50 - 100 cm', '100 - 150 cm',
            'precipitacao_7_dias', 'precipitacao_14_dias',
           'precipitação']
correlation = df_completo[columns].corr()#method='spearman'
plot = sns.heatmap(correlation, annot = True, fmt=".1f", linewidths=.6,cmap='cubehelix_r')
plot

In [ ]:
df_completo[columns].describe()

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss

def test_stationarity(series, name="serie"):
    print(f"--- Teste para {name} ---")

    # ADF
    adf_result = adfuller(series.dropna())
    print(f"ADF p-value: {adf_result[1]:.4f}")
    if adf_result[1] < 0.05:
        print("→ Rejeita H0: estacionária ✅")
    else:
        print("→ Não rejeita H0: NÃO estacionária ❌")

    # KPSS
    kpss_result = kpss(series.dropna(), regression='c', nlags="auto")
    print(f"KPSS p-value: {kpss_result[1]:.4f}")
    if kpss_result[1] > 0.05:
        print("→ Não rejeita H0: estacionária ✅")
    else:
        print("→ Não aceita H0: NÃO estacionária ❌")

    print("\n")

# Supondo df['media'] e df['precipitacao_14_dias']
test_stationarity(df_completo['100 - 150 cm'], "media")
test_stationarity(df_completo['precipitacao_14_dias'], "precipitacao_14_dias")

In [ ]:
df_completo['media_diff'] = df_completo['media'].diff()

In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests

max_lag = 5  # ajuste conforme necessário

# Teste: precipitação → causa → media
print("\nTeste: precipitacao_14_dias Granger-causa media?")
grangercausalitytests(df_completo[['media_diff', 'precipitação']][1:], maxlag=max_lag, verbose=True)

# Teste inverso: media → causa → precipitação
print("\nTeste: media Granger-causa precipitacao_14_dias?")
grangercausalitytests(df_completo[['precipitação', 'media']], maxlag=max_lag, verbose=True)

In [ ]:
fig = plt.figure(figsize=(12, 7))
gs = gridspec.GridSpec(2, 1, height_ratios=[2, 1])  # Define a proporção do espaço vertical (2/3 para o primeiro gráfico e 1/3 para o segundo gráfico)

ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1], sharex=ax1)

# Gráfico de Umidade do Solo
ax1.plot(df_frei_orestes_d['datahora'], df_frei_orestes_d["media"], color='black', label='0 - 50 cm')
#ax1.plot(df_frei_orestes_d['datahora'], df_frei_orestes_d["media"], color='yellow', label='50 - 100 cm')
#ax1.plot(df_frei_orestes_d['datahora'], df_frei_orestes_d["media"], color='red', label='100 - 150 cm')

ax1.set_ylabel('Umidade do solo (%)', color='black', fontsize=15)
ax1.set_title('Gráfico de Umidade do Solo e Precipitação', fontsize=16, fontweight='bold')
ax1.tick_params(axis='x', rotation=90)
ax1.grid(True)
# Gráfico de Precipitação
ax2.plot(df_frei_orestes_d['datahora'], df_frei_orestes_d['precipitação'], color='blue', alpha=0.5, label='Precipitação (mm)')

ax2.set_ylabel('Precipitação', color='black', fontsize=15)
ax2.tick_params(axis='y', labelcolor='black')
ax2.tick_params(axis='x', rotation=45)
ax2.grid(True)

locator = mdates.MonthLocator(interval=3)  # Define a cada 2 meses
formatter = mdates.DateFormatter('%Y-%m')

ax2.xaxis.set_major_locator(locator)
ax2.xaxis.set_major_formatter(formatter)
# Adicionando as legendas
fig.legend(loc='upper center', bbox_to_anchor=(0.5, -0.005), ncol=4, fontsize=12)

# Ajustando layout
fig.tight_layout()
fig.subplots_adjust(hspace=0.0)  # Reduzir o espaço vertical entre os subplots
plt.show()

In [ ]:
#!pip install statsmodels

In [ ]:
import matplotlib.pyplot as plt
import datetime
import pandas as pd
import numpy as np
import seaborn as sns
from pandas.plotting import register_matplotlib_converters
import statsmodels.api as sm
from statsmodels.tsa.seasonal import MSTL
from statsmodels.tsa.seasonal import DecomposeResult
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

In [ ]:
#'0 - 50 cm', '50 - 100 cm', '100 - 150 cm' 'precipitacao_14_dias'


In [ ]:
# Lista das colunas
colunas = ['0 - 50 cm', '50 - 100 cm', '100 - 150 cm', 'precipitacao_14_dias']

fig, axes = plt.subplots(len(colunas), 4, figsize=(16, 10), sharex=False)

# Loop pelas variáveis
for i, coluna in enumerate(colunas):
    serie = df_frei_orestes_d[coluna].dropna()

    # Decomposição MSTL
    mstl = MSTL(serie, periods=365)
    res = mstl.fit()

    tendencia = res.trend
    sazonalidade = res.seasonal
    residuos = serie - tendencia - sazonalidade

    # Série original
    axes[i, 0].plot(serie, color='black')
    axes[i, 0].set_title(f'{coluna}', fontsize=11)

    # Tendência
    axes[i, 1].plot(tendencia, color='black')
    axes[i, 1].set_title('Tendência', fontsize=11)

    # Sazonalidade
    axes[i, 2].plot(sazonalidade, color='black')
    axes[i, 2].set_title('Sazonalidade', fontsize=11)

    # Resíduos
    axes[i, 3].plot(residuos, color='black')
    axes[i, 3].set_title('Resíduos', fontsize=11)

# Layout geral
plt.tight_layout()
plt.show()

In [ ]:
plot_acf(df_frei_orestes_d['media'])
plot_pacf(df_frei_orestes_d['media'])
plt.show()

In [ ]:
dif_y = df_frei_orestes_d['media'].diff(periods=1).dropna()
plot_acf(dif_y)
plot_pacf(dif_y)
plt.show()

# separar dados -> train, test

In [ ]:
#sarimax
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from scipy.stats import shapiro, jarque_bera
from statsmodels.tsa.arima.model import ARIMA

#lstm
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
#nnar
from arnet import ARNet
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
#nbeats
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from darts import TimeSeries, concatenate

#tft
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")
import logging
logging.disable(logging.CRITICAL)

from darts.dataprocessing.transformers import Scaler
from darts.models import TFTModel, NaiveSeasonal, NaiveDrift, ExponentialSmoothing, BlockRNNModel, RNNModel, NBEATSModel
from darts.utils.statistics import check_seasonality, extract_trend_and_seasonality, plot_acf
from darts.datasets import AirPassengersDataset
from darts.utils.timeseries_generation import datetime_attribute_timeseries
from darts.utils.likelihood_models import QuantileRegression
from darts.utils.utils import ModelMode, SeasonalityMode, TrendMode

from darts.models import ARIMA
from darts.dataprocessing.transformers import BoxCox

from darts.metrics import mse, rmse, mae, mape

pd.set_option("display.precision",2)
np.set_printoptions(precision=2, suppress=True)
pd.options.display.float_format = '{:,.2f}'.format

In [ ]:
'''df_completo = pd.read_csv('/Users/Administrador/Documents/umidade_solo/df_frei_orestes_m.csv')
df_completo['datahora'] = pd.to_datetime(df_completo['datahora'])'''

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
#test_size = 1 - (1826 / len(df_completo))

# Series
y = df_completo['100 - 150 cm'].values.reshape(-1,1)
X = df_completo['precipitacao_14_dias'].values.reshape(-1,1)

y = np.log1p(y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
# Padronização (fit só no treino)
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled  = scaler_X.transform(X_test)

y_train_scaled = scaler_y.fit_transform(y_train)
y_test_scaled  = scaler_y.transform(y_test)

print("Shapes:")
print(X_train_scaled.shape, X_test_scaled.shape)
print(y_train_scaled.shape, y_test_scaled.shape)

In [ ]:
from darts.dataprocessing.transformers import Scaler
series = TimeSeries.from_dataframe(df_completo,
                                   time_col="datahora",
                                   value_cols="100 - 150 cm")
cov = TimeSeries.from_dataframe(df_completo,
                                time_col="datahora",
                                value_cols="precipitacao_14_dias")

# Split the data into train and test sets
train, remainder = series.split_before(0.6)#0.765
val, test = remainder.split_before(0.5)
cov_train, cov_remainder = cov.split_before(0.6)
cov_val, cov_test = cov_remainder.split_before(0.5)

scaler_yy = Scaler()
scaler_xx = Scaler()

train_scaled = scaler_yy.fit_transform(train)
val_scaled = scaler_yy.transform(val)
test_scaled = scaler_yy.transform(test)

cov_scaled = scaler_xx.fit_transform(cov)
cov_train_scaled = scaler_xx.transform(cov_train)
cov_val_scaled = scaler_xx.transform(cov_val)
cov_test_scaled = scaler_xx.transform(cov_test)

print("Shapes:")
print(train_scaled.shape, test_scaled.shape)
print(cov_train_scaled.shape, cov_test_scaled.shape)


In [ ]:
n_step = len(y_test_scaled)#10009

# SARIMAX

In [ ]:
from darts.dataprocessing.transformers import Scaler
series = TimeSeries.from_dataframe(df_completo,
                                   time_col="datahora",
                                   value_cols="0 - 50 cm")
cov = TimeSeries.from_dataframe(df_completo,
                                time_col="datahora",
                                value_cols="precipitacao_14_dias")

y_train, y_test = series.split_before(0.8)#0.765
X_train, X_test = cov.split_before(0.8)

#scaler_X = BoxCox()
#scaler_y = BoxCox()
scaler_X = Scaler()
scaler_y = Scaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled  = scaler_X.transform(X_test)

y_train_scaled = scaler_y.fit_transform(y_train)
y_test_scaled  = scaler_y.transform(y_test)

print("Shapes:")
print(X_train_scaled.shape, X_test_scaled.shape)
print(y_train_scaled.shape, y_test_scaled.shape)

In [ ]:
# Define as ordens do modelo
# 50 cm  (1,1,2)(0,1,0)[365]  3.1.0#-250.263  (2,1,0) -4080.316        sem exo -> (0,1,3) 4896.820
# 100 cm (2,1,0)(0,1,0)[365] # 100cm -790.096 (2,1,0) # -6360.496                   (2,1,1)
# 150 cm (4,1,0)(0,1,0)[365] (2,1,0) -7480.822                                       (5,1,1)  (0,1,3) -7479.48
# media  (3,1,0)(0,1,0)[365]
results_sarimax = ARIMA(p=1, d=1, q=2, seasonal_order=(0, 1, 0, 365))
start = time.time()
results_sarimax.fit(y_train_scaled)
end = time.time()
tempo_sarimax = end - start



In [ ]:
results_sarimax.model.summary()

In [ ]:
print("\n=== Teste Ljung-Box (Autocorrelação) -> (>0.05) resíduos sem autocorrelação ===")
ljung = acorr_ljungbox(results_sarimax.model.resid, lags=[10, 20, 30], return_df=True)
print(ljung)
print("\n=== Teste de Normalidade -> (>0.05) resíduos parecem normais ===")
sw = shapiro(results_sarimax.model.resid)
jb = jarque_bera(results_sarimax.model.resid)
print(f"Shapiro-Wilk p-value: {sw.pvalue:.4f}")
print(f"Jarque-Bera p-value: {jb.pvalue:.4f}")
arch_test = het_arch(results_sarimax.model.resid) #homoscedasticity
print("\n=== ARCH Test -> (>0.05) resíduos homoscedásticos ===")
print(f"LM stat: {arch_test[0]:.4f}, p-valor: {arch_test[1]:.4f}\n")

In [ ]:
forecast_sarimax = results_sarimax.predict(n_step)
forecast_sarimax  = scaler_y.inverse_transform(forecast_sarimax)#forecast_sarimax.reshape(-1,1))

# Optional: Plot the forecast
plt.figure(figsize=(12, 6))
y_test.plot(label='Observed')
forecast_sarimax.plot(color='red', label='Forecast')
plt.title('SARIMAX Forecast')
plt.xlabel('Time')
plt.ylabel('Media Value')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Calcular métricas
mse_sarimax = mse(y_test, forecast_sarimax)
rmse_sarimax = rmse(y_test, forecast_sarimax)
mae_sarimax = mae(y_test, forecast_sarimax)
mape_sarimax = mape(y_test, forecast_sarimax)

print(f"MSE: {mse_sarimax:.4f}")
print(f"RMSE: {rmse_sarimax:.4f}")
print(f"MAE: {mae_sarimax:.4f}")
print(f"MAPE: {mape_sarimax:.2f}%")

#NNARX

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
#test_size = 1 - (1826 / len(df_completo))

# Series
y = df_completo['100 - 150 cm'].values.reshape(-1,1)
X = df_completo['precipitacao_14_dias'].values.reshape(-1,1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
# Padronização (fit só no treino)
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled  = scaler_X.transform(X_test)

y_train_scaled = scaler_y.fit_transform(y_train)
y_test_scaled  = scaler_y.transform(y_test)

print("Shapes:")
print(X_train_scaled.shape, X_test_scaled.shape)
print(y_train_scaled.shape, y_test_scaled.shape)

In [ ]:
model_nnar = ARNet(base_model=MLPRegressor(hidden_layer_sizes=(200), activation='relu', max_iter=2000), #  por padrão -> hidden_layer_sizes=(100), activation='relu', max_iter=2000
    p = 365,           # número de lags autorregressivos
    P = 1,           # número de lags sazonais
    seasonality = 365,
    repeats = 20     # média de  20 redes treinada (como em nnetar)
)
start = time.time()

model_nnar.fit(y_train_scaled.reshape(-1))#, X=X_train_scaled)#.reshape(-1))#

end = time.time()
tempo_nnar = end - start

In [ ]:
forecast_nnar = model_nnar.predict(n_steps=n_step)#, X=X_test_scaled[:n_step])
forecast_nnar = scaler_y.inverse_transform(forecast_nnar.reshape(-1,1))

In [ ]:
plt.plot(y_test)
plt.plot(forecast_nnar)

In [ ]:
# Calculate MSE
mse_nnar = mean_squared_error(y_test, forecast_nnar)
print(f"Mean Squared Error (MSE): {mse_nnar:.2f}")

# Calculate RMSE
rmse_nnar = np.sqrt(mse_nnar)
print(f"Root Mean Squared Error (RMSE): {rmse_nnar:.2f}")

# Calculate MAE
mae_nnar = mean_absolute_error(y_test, forecast_nnar)
print(f"Mean Absolute Error (MAE): {mae_nnar:.2f}")

# Calculate MAPE
# Avoid division by zero if actual values are zero
mape_nnar = np.mean(np.abs((y_test - forecast_nnar) / y_test)) * 100
print(f"Mean Absolute Percentage Error (MAPE): {mape_nnar:.2f}%")

# lstm Darts

In [ ]:
from darts.dataprocessing.transformers import Scaler
series = TimeSeries.from_dataframe(df_completo,
                                   time_col="datahora",
                                   value_cols="100 - 150 cm")
cov = TimeSeries.from_dataframe(df_completo,
                                time_col="datahora",
                                value_cols="precipitacao_14_dias")

# Split the data into train and test sets
train, remainder = series.split_before(0.6)#0.765
val, test = remainder.split_before(0.5)
cov_train, cov_remainder = cov.split_before(0.6)
cov_val, cov_test = cov_remainder.split_before(0.5)

scaler_yy = Scaler()
scaler_xx = Scaler()

train_scaled = scaler_yy.fit_transform(train)
val_scaled = scaler_yy.transform(val)
test_scaled = scaler_yy.transform(test)

cov_scaled = scaler_xx.fit_transform(cov)
cov_train_scaled = scaler_xx.transform(cov_train)
cov_val_scaled = scaler_xx.transform(cov_val)
cov_test_scaled = scaler_xx.transform(cov_test)

print("Shapes:")
print(train_scaled.shape, test_scaled.shape)
print(cov_train_scaled.shape, cov_test_scaled.shape)

In [ ]:
from pytorch_lightning.callbacks import Callback

class LossRecorder(Callback):
    def __init__(self):
        self.train_loss_history = []
        self.val_loss_history = []

    def on_train_epoch_end(self, trainer, pl_module):
        self.train_loss_history.append(trainer.callback_metrics["train_loss"].item())
        self.val_loss_history.append(trainer.callback_metrics["val_loss"].item())
loss_recorder = LossRecorder()

In [ ]:
model_lstm = BlockRNNModel(
    input_chunk_length=30,#30
    output_chunk_length=1,#1
    model="LSTM",
    hidden_dim=30,# 100 e 150cm -> 30, 50cm -> 20
    #n_rnn_layers=2,
    dropout=0.1,#0
    batch_size=16,#16
    n_epochs=50,#100
    optimizer_kwargs={"lr": 1e-3},
    model_name="Air_RNN",
    log_tensorboard=True,
    random_state=42,
    force_reset=True,
    use_static_covariates=True,
    #add_encoders=add_encoders,
    pl_trainer_kwargs={"callbacks": [loss_recorder]},
    save_checkpoints=True
)

In [ ]:
start = time.time()
model_lstm.fit(
    train_scaled,
    #past_covariates=cov_train_scaled,
    val_series=val_scaled,
    #val_past_covariates=cov_val_scaled,
    #val_future_covariates=cov_scaled[['sin_ano', 'cos_ano']],
    #future_covariates=cov_scaled['sin_ano'],
    verbose=True,
)

end = time.time()
tempo_lstm = end - start

In [ ]:
forecast_lstm = model_lstm.predict(n=830)#, past_covariates=cov_scaled)#, future_covariates=cov_scaled['sin_ano'])
forecast_lstm = scaler_yy.inverse_transform(forecast_lstm)
plt.figure(figsize=(8, 5))
remainder.plot(label="test real")
forecast_lstm.plot(label="forecast")
plt.title(f"MAPE: {mape(forecast_lstm, test):.2f}%")
plt.legend()
plt.show()

In [ ]:
plt.plot(loss_recorder.train_loss_history, label='Train Loss')
plt.plot(loss_recorder.val_loss_history, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
'''historical_forecasts_train = model_lstm.historical_forecasts(
    series=train_scaled,
    past_covariates=cov_scaled,
    start=train_scaled.time_index[model_lstm.input_chunk_length],
    forecast_horizon=1,
    stride=1,
    retrain=False,
    verbose=True,
    last_points_only=False,
)
datas = [ts.time_index[0] for ts in historical_forecasts_train]
valores = [ts.values().item() for ts in historical_forecasts_train]
df_forecasts = pd.DataFrame({
    "datahora": datas,
    "forecast": valores
})
historical_forecasts_train = TimeSeries.from_dataframe(df_forecasts,
                                time_col="datahora",
                                value_cols="forecast")'''

In [ ]:
# Calcular métricas
mse_lstm = mse(test[:n_step], forecast_lstm[len(cov_test):])
rmse_lstm = rmse(test[:n_step], forecast_lstm[len(cov_test):])
mae_lstm = mae(test[:n_step], forecast_lstm[len(cov_test):])
mape_lstm = mape(test[:n_step], forecast_lstm[len(cov_test):])

print(f"MSE: {mse_lstm:.4f}")
print(f"RMSE: {rmse_lstm:.4f}")
print(f"MAE: {mae_lstm:.4f}")
print(f"MAPE: {mape_lstm:.2f}%")

# n-beats

In [ ]:
'''# Initialize the N-BEATS model
model_nbeats = NBEATSModel(input_chunk_length=365, output_chunk_length=1,  n_epochs=20, random_state=42)
# Fit the model
model_nbeats.fit(train_scaled)#, past_covariates=cov_scaled) #val_series=val,val_past_covariates=cov,

forecast_nbeats = model_nbeats.predict(n=n_step)#, past_covariates=cov_scaled)
forecast_nbeats = scaler_yy.inverse_transform(forecast_nbeats)
# ===========================
# 6. Visualização
# ===========================
plt.figure(figsize=(12, 6))
test.plot(label="Série real")
forecast_nbeats.plot(label="Previsão")
plt.title("Previsão N-BEATS ")
plt.legend()
plt.show()

# Calcular métricas
mse_nbeats = mse(test, forecast_nbeats)
rmse_nbeats = rmse(test, forecast_nbeats)
mae_nbeats = mae(test, forecast_nbeats)
mape_nbeats = mape(test, forecast_nbeats)

print(f"MSE: {mse_nbeats:.4f}")
print(f"RMSE: {rmse_nbeats:.4f}")
print(f"MAE: {mae_nbeats:.4f}")
print(f"MAPE: {mape_nbeats:.2f}%")'''

# TFT

In [ ]:
EPOCHS = 10
INLEN = 32
HIDDEN = 64
LSTMLAYERS = 2
ATTHEADS = 1
DROPOUT = 0.1
BATCH = 32

N_FC = 36           # default forecast horizon
RAND = 42           # set random state
N_SAMPLES = 100     # number of times a prediction is sampled from a probabilistic model
N_JOBS = 3          # parallel processors to use;  -1 = all processors

# default quantiles for QuantileRegression
QUANTILES = [0.01, 0.05, 0.1, 0.2, 0.25, 0.5, 0.75, 0.8, 0.9, 0.95, 0.99]

MSEAS = 120          # max seasonality to check: days
ALPHA = 0.05        # significance level for seasonality test
FIGSIZE = (9, 6)

qL1, qL2, qL3 = 0.01, 0.05, 0.10        # percentiles of predictions: lower bounds
qU1, qU2, qU3 = 1-qL1, 1-qL2, 1-qL3     # upper bounds derived from lower bounds
label_q1 = f'{int(qU1 * 100)} / {int(qL1 * 100)} percentile band'
label_q2 = f'{int(qU2 * 100)} / {int(qL2 * 100)} percentile band'
label_q3 = f'{int(qU3 * 100)} / {int(qL3 * 100)} percentile band'

In [ ]:
model_tft = TFTModel(input_chunk_length=INLEN,
                    output_chunk_length=N_FC,
                    hidden_size=HIDDEN,
                    lstm_layers=LSTMLAYERS,
                    num_attention_heads=ATTHEADS,
                    dropout=DROPOUT,
                    batch_size=BATCH,
                    n_epochs=EPOCHS,
                    #add_relative_index=True,
                    #add_relative_index=false # prede covariaveis futuras
                    likelihood=QuantileRegression(quantiles=QUANTILES),
                    # loss_fn=MSELoss(),
                    random_state=RAND,
                    force_reset=True)

In [ ]:
# training
model_tft.fit(train_scaled,
            future_covariates=cov_scaled,
            verbose=True)

In [ ]:
# testing: generate predictions
forecast_tft = model_tft.predict(n=len(test),
                            num_samples=N_SAMPLES,
                            future_covariates=cov_scaled,
                            n_jobs=N_JOBS)
forecast_tft = scaler_yy.inverse_transform(forecast_tft)

In [ ]:
forecast_scaled = model_tft.predict(
    series=train_scaled,  # série base até o final do treino
    future_covariates=cov_scaled,
    n=31,                 # número de passos à frente (dias)
)

In [ ]:
# testing: helper function: plot predictions
def plot_predict(ts_actual, ts_test, ts_pred):

    ## plot time series, limited to forecast horizon
    plt.figure(figsize=FIGSIZE)

    ts_actual.plot(label="actual")                                       # plot actual

    ts_pred.plot(low_quantile=qL1, high_quantile=qU1, label=label_q1)    # plot U1 quantile band
    #ts_pred.plot(low_quantile=qL2, high_quantile=qU2, label=label_q2)   # plot U2 quantile band
    ts_pred.plot(low_quantile=qL3, high_quantile=qU3, label=label_q3)    # plot U3 quantile band
    ts_pred.plot(central_quantile="mean", label="expected")              # plot "mean" or median=0.5

    plt.title("TFT: test set (MAPE: {:.2f}%)".format(mape(ts_test, ts_pred)))
    plt.legend();


In [ ]:
# testing: call helper function: plot predictions
#ts_tpred_s1 = transformer.inverse_transform(ts_tpred1)
plot_predict(series, test, forecast_tft)

In [ ]:
#ts_pred1 = transformer.inverse_transform(ts_tpred1)
ts_actual1 = series[ test.start_time(): test.end_time() ]  # actual values in forecast horizon
plot_predict(ts_actual1, test, forecast_tft)

In [ ]:
from darts.explainability import TFTExplainer

In [ ]:
explainer = TFTExplainer(model_tft)
explainability_result = explainer.explain()
explainer.plot_variable_selection(explainability_result)

In [ ]:
mse_tft = mse(test, forecast_tft)
rmse_tft = rmse(test, forecast_tft)
mae_tft = mae(test, forecast_tft)
mape_tft = mape(test, forecast_tft)

print(f"MSE: {mse_tft:.4f}")
print(f"RMSE: {rmse_tft:.4f}")
print(f"MAE: {mae_tft:.4f}")
print(f"MAPE: {mape_tft:.2f}%")

# previsão + metricas




In [ ]:
n_step = len(y_test)

## SARIMAX

In [ ]:
# sarimax
start_index = len(X_train)
end_index = start_index + n_step - 1

# Generate the forecast with confidence intervals
forecast_sarimax = results_sarimax.predict(start=start_index, end=end_index, exog=X_test_scaled[:n_step])
forecast_sarimax = scaler_y.inverse_transform(forecast_sarimax.reshape(-1,1))
# Optional: Plot the forecast
plt.figure(figsize=(12, 6))
plt.plot(df_completo.index, df_completo['media'], label='Observed')
plt.plot(forecast_sarimax, color='red', label='Forecast')
plt.title('SARIMAX Forecast')
plt.xlabel('Time')
plt.ylabel('Media Value')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Calculate MSE
mse_sarimax = mean_squared_error(y_test[:n_step], forecast_sarimax)
print(f"Mean Squared Error (MSE): {mse_sarimax:.2f}")

# Calculate RMSE
rmse_sarimax = np.sqrt(mse_sarimax)
print(f"Root Mean Squared Error (RMSE): {rmse_sarimax:.2f}")

# Calculate MAE
mae_sarimax = mean_absolute_error(y_test[:n_step], forecast_sarimax)
print(f"Mean Absolute Error (MAE): {mae_sarimax:.2f}")

# Calculate MAPE
# Avoid division by zero if actual values are zero
mape_sarimax = np.mean(np.abs((y_test[:n_step] - forecast_sarimax) / y_test[:n_step])) * 100
print(f"Mean Absolute Percentage Error (MAPE): {mape_sarimax:.2f}%")

## NNAR

In [ ]:
#nnar
forecast_nnar = model_nnar.predict(n_steps=n_step, X=X_test_scaled[0:n_step])
forecast_nnar = scaler_y.inverse_transform(forecast_nnar.reshape(-1,1))
plt.plot(forecast_nnar)
plt.plot(y_test)

In [ ]:
# Calculate MSE
mse_nnar = mean_squared_error(y_test[:n_step], forecast_nnar)
print(f"Mean Squared Error (MSE): {mse_nnar:.2f}")

# Calculate RMSE
rmse_nnar = np.sqrt(mse_nnar)
print(f"Root Mean Squared Error (RMSE): {rmse_nnar:.2f}")

# Calculate MAE
mae_nnar = mean_absolute_error(y_test[:n_step], forecast_nnar)
print(f"Mean Absolute Error (MAE): {mae_nnar:.2f}")

# Calculate MAPE
# Avoid division by zero if actual values are zero
mape_nnar = np.mean(np.abs((y_test[:n_step] - forecast_nnar) / y_test[:n_step])) * 100
print(f"Mean Absolute Percentage Error (MAPE): {mape_nnar:.2f}%")

## LSTM

In [ ]:
forecast_lstm = model_lstm.predict(n=n_step, past_covariates=cov_scaled)
forecast_lstm = scaler_yy.inverse_transform(forecast_lstm)


In [ ]:
plt.figure(figsize=(12, 6))
series.plot(label="Série real")
forecast_lstm.plot(label="Previsão 300 passos à frente")
plt.title("Previsão LSTM (300 passos futuros)")
plt.legend()
plt.show()

In [ ]:
# Calcular métricas
mse_lstm = mse(test[:n_step], forecast_lstm)
rmse_lstm = rmse(test[:n_step], forecast_lstm)
mae_lstm = mae(test[:n_step], forecast_lstm)
mape_lstm = mape(test[:n_step], forecast_lstm)

print(f"MSE: {mse_lstm:.4f}")
print(f"RMSE: {rmse_lstm:.4f}")
print(f"MAE: {mae_lstm:.4f}")
print(f"MAPE: {mape_lstm:.2f}%")

## N-BEATS

In [ ]:
'''forecast_nbeats = model_nbeats.predict(n=n_step, past_covariates=cov_scaled)
forecast_nbeats = scaler_yy.inverse_transform(forecast_nbeats)
# ===========================
# 6. Visualização
# ===========================
plt.figure(figsize=(12, 6))
series.plot(label="Série real")
forecast_nbeats.plot(label="Previsão XXX passos à frente")
plt.title("Previsão N-BEATS ")
plt.legend()
plt.show()

# Calcular métricas
mse_nbeats = mse(test[:n_step], forecast_nbeats)
rmse_nbeats = rmse(test[:n_step], forecast_nbeats)
mae_nbeats = mae(test[:n_step], forecast_nbeats)
mape_nbeats = mape(test[:n_step], forecast_nbeats)

print(f"MSE: {mse_nbeats:.4f}")
print(f"RMSE: {rmse_nbeats:.4f}")
print(f"MAE: {mae_nbeats:.4f}")
print(f"MAPE: {mape_nbeats:.2f}%")'''

## TFT

In [ ]:
'''forecast_tft = model_tft.predict(n=n_step,
                            num_samples=N_SAMPLES,
                            future_covariates=cov_scaled,
                            n_jobs=N_JOBS)
forecast_tft = scaler_yy.inverse_transform(forecast_tft)
ts_actual1 = series[test.start_time(): test.end_time() ]  # actual values in forecast horizon
plot_predict(ts_actual1, test, forecast_tft)
mse_tft = mse(test, forecast_tft)
rmse_tft = rmse(test, forecast_tft)
mae_tft = mae(test, forecast_tft)
mape_tft = mape(test, forecast_tft)

print(f"MSE: {mse_tft:.4f}")
print(f"RMSE: {rmse_tft:.4f}")
print(f"MAE: {mae_tft:.4f}")
print(f"MAPE: {mape_tft:.2f}%")'''

# Metricas

In [ ]:
plt.figure(figsize=(12,6))  # tamanho da figura

plt.plot(df_completo['datahora'][-len(y_test):], y_test, label="Dados reais", linewidth=2)
plt.plot(df_completo['datahora'][-len(y_test)-1:], forecast_lstm.values()[len(cov_test):], label="LSTM", linestyle='--')
#plt.plot( forecast_tft.mean(axis=2).values(), label="TFT", linestyle='--')
plt.plot(df_completo['datahora'][-len(y_test):], forecast_sarimax, label="SARIMAX", linestyle='--')
#plt.plot( forecast_nbeats.mean(axis=2).values(), label="NBEATS", linestyle='--')
plt.plot(df_completo['datahora'][-len(y_test):], forecast_nnar, label="NNAR", linestyle='--')

#plt.title("Comparação das Previsões")
plt.xlabel("Data")
plt.ylabel("Umidade do Solo (%)")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
metrics = {
    "SARIMAX": {
        "MSE": mse_sarimax, "RMSE": rmse_sarimax, "MAE": mae_sarimax, "MAPE": mape_sarimax,
        "Tempo (s)": tempo_sarimax
    },
    "NNAR": {
        "MSE": mse_nnar, "RMSE": rmse_nnar, "MAE": mae_nnar, "MAPE": mape_nnar,
        "Tempo (s)": tempo_nnar
    },
    "LSTM": {
        "MSE": mse_lstm, "RMSE": rmse_lstm, "MAE": mae_lstm, "MAPE": mape_lstm,
        "Tempo (s)": tempo_lstm
    }
}

# Printar de forma organizada
print(f"{'Modelo':<10} {'MSE':<10} {'RMSE':<10} {'MAE':<10} {'MAPE':<10} {'Tempo (s)':<10}")
for model, vals in metrics.items():
    print(f"{model:<10} {vals['MSE']:<10.2f} {vals['RMSE']:<10.2f} {vals['MAE']:<10.2f} {vals['MAPE']:<10.2f} {vals['Tempo (s)']:<10.2f}")

#Simulação


In [ ]:
# concaternar test ate 31/21/2024 com dados api
# prever
# metricas

In [ ]:
simulacao = pd.read_csv('/Users/Administrador/Downloads/precipitacao_merge_2025_01.csv')
simulacao['data'] = pd.to_datetime(simulacao['data'])
simulacao = simulacao.sort_values(by='data')
simulacao = simulacao.reset_index(drop=True)

simulacao['precipitacao_14_dias'] = 0.0
janela_60 = 14
simulacao['precipitacao_14_dias'] = simulacao['precipitacao'].rolling(window=janela_60, min_periods=1).sum()


In [ ]:
simulacao['ano'] = simulacao['data'].dt.year
resumo_anual = simulacao.groupby('ano')['precipitacao_14_dias'].describe()
resumo_anual

In [ ]:
simulacao_13 = simulacao[(simulacao['data'] >= '2014-07-14') &
                         (simulacao['data'] <= '2015-08-31')].copy()
simulacao_11 = simulacao[(simulacao['data'] >= '2008-07-14') &
                         (simulacao['data'] <= '2009-08-31')].copy()

In [ ]:
plt.plot(simulacao_13['data'], simulacao_13['precipitacao'])
plt.plot(simulacao_11['data'],simulacao_11['precipitacao'])

In [ ]:
cov_13 = scaler_X.transform(simulacao_13['precipitacao_14_dias'].values.reshape(-1,1)*0.8)
cov_11 = scaler_X.transform(simulacao_11['precipitacao_14_dias'].values.reshape(-1,1)*1.2)

forecast_nnar_13 = model_nnar.predict(n_steps=n_step, X=cov_13)
forecast_nnar_13 = scaler_y.inverse_transform(forecast_nnar_13.reshape(-1,1))
forecast_nnar_11 = model_nnar.predict(n_steps=n_step, X=cov_11)
forecast_nnar_11 = scaler_y.inverse_transform(forecast_nnar_11.reshape(-1,1))




In [ ]:
plt.plot(forecast_nnar_13, label='13')
plt.plot(forecast_nnar_11, label='11')
plt.plot(y_test, label='real')
plt.legend()
plt.show()

In [ ]:
simulacao = simulacao[simulacao['data'] > pd.Timestamp('2024-07-13')]
X_simulacao_scaled  = scaler_X.transform(simulacao['precipitacao_14_dias'].values.reshape(-1, 1))

# Graficos


In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 3))

# Gráfico de Umidade do Solo
ax1.plot(df_frei_orestes_d['datahora'], df_frei_orestes_d["0 - 50 cm"], color='black', label='0 - 50 cm')

ax1.set_ylabel('Umidade do solo (%)', color='black', fontsize=15)
ax1.tick_params(axis='x', rotation=45)
ax1.grid(True)


locator = mdates.MonthLocator(interval=3)  # Define a cada 2 meses
formatter = mdates.DateFormatter('%Y-%m')

ax1.xaxis.set_major_locator(locator)
ax1.xaxis.set_major_formatter(formatter)

# Adicionando as legendas
fig.legend(loc='upper center', bbox_to_anchor=(0.5, -0.005), ncol=4, fontsize=12)

# Ajustando layout
fig.tight_layout()
plt.show()

In [ ]:
#2020-01-01 ... 2024-07-12 # 2024-07-13 ... 2025-08-31

In [ ]:
from datetime import datetime

# Intervalo de anos e meses
anos = range(2022, 2026)
meses = range(1, 13)

# Gerar URLs
urls = []
for ano in anos:
    #print(ano)
    for mes in meses:
        # Parar em agosto de 2025
        if ano == 2025 and mes > 8:
            break

        url = f"https://ftp.cptec.inpe.br/modelos/tempo/MERGE/GPM/DAILY/{ano}/{mes:02d}/"
        urls.append(url)

In [ ]:
import requests
from bs4 import BeautifulSoup
import os

for url in urls:
    # URL da página com os arquivos
    #url = "https://ftp.cptec.inpe.br/modelos/tempo/MERGE/GPM/DAILY/2024/04/"

    # Fazer requisição HTTP
    response = requests.get(url)
    response.raise_for_status()
    # Parsear o HTML da página
    soup = BeautifulSoup(response.text, "html.parser")

    # Encontrar todos os links da página
    links = soup.find_all("a")
    # Criar pasta local para salvar os arquivos
    os.makedirs("/Users/Administrador/Downloads/MERGE_2000_2025", exist_ok=True)

    # Percorrer os links e baixar apenas os arquivos desejados
    for link in links:
        href = link.get("href")
        if href and href.endswith((".grib2", ".ctl", ".idx")):  # filtra tipos de arquivo
            file_url = url + href
            local_path = os.path.join("/Users/Administrador/Downloads/MERGE_2000_2025", href)

            print(f"Baixando {href}...")
            file_response = requests.get(file_url, stream=True)
            with open(local_path, "wb") as f:
                for chunk in file_response.iter_content(chunk_size=8192):
                    f.write(chunk)

    print("✅ Todos os arquivos foram baixados!")

In [ ]:
url = "https://ftp.cptec.inpe.br/modelos/tempo/MERGE/GPM/DAILY/2014/12/"
response = requests.get(url)
response.raise_for_status()
# Parsear o HTML da página
soup = BeautifulSoup(response.text, "html.parser")

# Encontrar todos os links da página
links = soup.find_all("a")
# Criar pasta local para salvar os arquivos
os.makedirs("/Users/Administrador/Downloads/MERGE_2000_2025", exist_ok=True)

# Percorrer os links e baixar apenas os arquivos desejados
for link in links:
    href = link.get("href")
    if href and href.endswith((".grib2", ".ctl", ".idx")):  # filtra tipos de arquivo
        file_url = url + href
        local_path = os.path.join("/Users/Administrador/Downloads/MERGE_2000_2025", href)

        print(f"Baixando {href}...")
        file_response = requests.get(file_url, stream=True)
        with open(local_path, "wb") as f:
            for chunk in file_response.iter_content(chunk_size=8192):
                f.write(chunk)

print("✅ Todos os arquivos foram baixados!")

# LSTM puro



In [ ]:
df_frei_orestes_d.columns

In [ ]:
def create_sequences_with_future_exog(y, X_exog, lookback, horizon):
    """
    Cria sequências para LSTM com:
    - Histórico de y (target) e X_exog (covariáveis)
    - Futuro CONHECIDO de X_exog até t+horizon
    - Saída: y futuro
    Parâmetros:
        y: vetor (n,)
        X_exog: matriz (n, k)
        lookback: janelas passadas
        horizon: passos futuros para prever
    """
    y = np.array(y).reshape(-1,1)
    X_exog = np.array(X_exog)

    n = len(y)
    n_features = X_exog.shape[1]
    X, y_out = [], []

    for i in range(n - lookback - horizon):
        # Passado: y + exógenas
        past_y = y[i:i+lookback]                     # (lookback, 1)
        past_exog = X_exog[i:i+lookback]            # (lookback, k)
        past_window = np.hstack([past_y, past_exog]) # (lookback, 1+k)

        # Futuro conhecido das exógenas
        future_exog = X_exog[i+lookback:i+lookback+horizon]  # (horizon, k)
        future_y_dummy = np.zeros((horizon,1))                # y futuro desconhecido
        future_window = np.hstack([future_y_dummy, future_exog])

        # Combina janelas (passado + futuro)
        combined = np.vstack([past_window, future_window])

        X.append(combined)
        y_out.append(y[i+lookback:i+lookback+horizon, 0])

    return np.array(X), np.array(y_out)


In [ ]:
horizon = 15
lookback = 30
X, y_future = create_sequences_with_future_exog(
    y=df_completo["media"].values,
    X_exog=df_completo[['precipitação', 'precipitacao_14_dias']].values,
    lookback=lookback,
    horizon=horizon
)

print("X shape:", X.shape)   # (amostras, timesteps, features)
print("y shape:", y_target.shape)

In [ ]:
# -------------------------------
# 4. Split treino/teste
# -------------------------------
train_size = int(0.8 * len(X))
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y_target[:train_size], y_target[train_size:]

In [ ]:
# -------------------------------
# 5. Modelo LSTM
# -------------------------------
model = Sequential([
    LSTM(128, input_shape=(X.shape[1], X.shape[2]), return_sequences=False),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(horizon)  # prever apenas y
])

model.compile(optimizer='adam', loss='mse')
model.summary()


In [ ]:
# -------------------------------
# 6. Treinar
# -------------------------------
history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=20,
    batch_size=32,
    verbose=1
)

In [ ]:
y_train.shape

In [ ]:
y_pred.shape

In [ ]:
y_pred = model.predict(X_train)
#inv_y_pred_train = model.predict(X_train)

# Inverter escala apenas para y
'''inv_y_pred_train = scaler.inverse_transform(
    np.concatenate([y_pred, np.zeros((len(y_pred), X.shape[2] -1))], axis=1))
inv_y_train = scaler.inverse_transform(
    np.concatenate([y_train, np.zeros((len(y_train), X.shape[2] -1))], axis=1))
'''
plt.figure(figsize=(10, 4))
plt.plot(y_train[:, 0], label='Real')
plt.plot(y_pred[:, 0], label='Previsto', alpha=0.7)
plt.title("Previsão de y (5 dias à frente) com exógenas futuras conhecidas")
plt.legend()
plt.show()

In [ ]:
y_pred.shape

In [ ]:
# -------------------------------
# 7. Avaliar e plotar resultados
# -------------------------------
y_pred = model.predict(X_test)


plt.figure(figsize=(10, 4))
plt.plot(y_test, label='Real')
plt.plot(y_pred[0,:], label='Previsto', alpha=0.7)
plt.title("Previsão de y ")
plt.legend()
plt.show()

In [ ]:
print(inv_y_pred_train.shape, y_pred.shape)

In [ ]:
len(inv_y_pred_train)

In [ ]:
trainPredictPlot = np.empty_like(df['media'])
trainPredictPlot[:] = np.nan
trainPredictPlot[lookback : len(inv_y_pred_train)+lookback] = inv_y_pred_train[:, 0]

testPredictPlot = np.empty_like(df['media'])
testPredictPlot[:] = np.nan
testPredictPlot[len(df['media']) - inv_y_pred_test.shape[0] : len(df['media'])] = inv_y_pred_test[:, 0]

In [ ]:
plt.plot(df_frei_orestes_d['datahora'], df['media'])
plt.plot(df_frei_orestes_d['datahora'], trainPredictPlot)
plt.plot(df_frei_orestes_d['datahora'], testPredictPlot)
plt.show()

In [ ]:
# Generate predictions for the next 100 time steps
predictions = []
current_sequence = X[-1].reshape((1, n_steps, 1))
for _ in range(100):
    prediction = model.predict(current_sequence, verbose=0)
    predictions.append(prediction[0])
    current_sequence = np.concatenate([current_sequence[:, 1:, :], [prediction]], axis=1)


# Print the predictions
print(predictions)

In [ ]:
mse_lstm = 0
rmse_lstm  = 0
mae_lstm  = 0
mape_lstm  = 0

# statsmodels

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
#test_size = 1 - (1826 / len(df_completo))

# Series
y = df_completo['100 - 150 cm'].values.reshape(-1,1)
X = df_completo['precipitacao_14_dias'].values.reshape(-1,1)


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
# Padronização (fit só no treino)
#scaler_X = StandardScaler()
#scaler_y = StandardScaler()
y_train = np.log1p(y_train)
X_train_scaled = X_train#scaler_X.fit_transform(X_train)
X_test_scaled  = X_test#scaler_X.transform(X_test)

y_train_scaled = y_train#scaler_y.fit_transform(y_train)
y_test_scaled  = y_test#scaler_y.transform(y_test)

print("Shapes:")
print(X_train_scaled.shape, X_test_scaled.shape)
print(y_train_scaled.shape, y_test_scaled.shape)

In [ ]:
# Define as ordens do modelo
# 50 cm  (1,1,2)(0,1,0)[365]  3.1.0#-250.263  (2,1,0) -4080.316        sem exo -> (0,1,3) -4060.912
# 100 cm (2,1,0)(0,1,0)[365] # 100cm -790.096 (2,1,0) # -6360.496                   (2,1,1) 6264.463
# 150 cm (4,1,0)(0,1,0)[365] (2,1,0) -7480.822                                       (3,1,1)  7476.365 (0,1,3) -7479.48
# media  (3,1,0)(0,1,0)[365]
order = (2,1,0)
seasonal_order = (0, 1, 0, 365)
# Cria o modelo SARIMAX com variável exógena
model_sarimax = SARIMAX(y_train_scaled,  order=order, seasonal_order=seasonal_order)#exog=X_train_scaled,

In [ ]:
# Ajusta o modelo aos dados
start = time.time()
results_sarimax = model_sarimax.fit(disp=False)
end = time.time()
tempo_sarimax = end - start


# Mostra o resumo dos resultados
print(results_sarimax.summary())

In [ ]:
print("\n=== Teste Ljung-Box (Autocorrelação) -> (>0.05) resíduos sem autocorrelação ===")
ljung = acorr_ljungbox(results_sarimax.resid, lags=[10, 20, 30], return_df=True)
print(ljung)
print("\n=== Teste de Normalidade -> (>0.05) resíduos parecem normais ===")
sw = shapiro(results_sarimax.resid)
jb = jarque_bera(results_sarimax.resid)
print(f"Shapiro-Wilk p-value: {sw.pvalue:.4f}")
print(f"Jarque-Bera p-value: {jb.pvalue:.4f}")
arch_test = het_arch(results_sarimax.resid) #homoscedasticity
print("\n=== ARCH Test -> (>0.05) resíduos homoscedásticos ===")
print(f"LM stat: {arch_test[0]:.4f}, p-valor: {arch_test[1]:.4f}\n")

In [ ]:
# Plotting residuals against fitted values to check for homoscedasticity
plt.figure(figsize=(10, 6))
plt.scatter(results_sarimax.fittedvalues, results_sarimax.resid)
plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs Fitted Values (Homoscedasticity Check)")
plt.axhline(0, color='red', linestyle='--')
plt.grid(True)
plt.show()


In [ ]:
# Plot the residuals
plt.figure(figsize=(10, 6))
plt.plot(results_sarimax.resid)
plt.title('SARIMA Residuals')
plt.xlabel('Time')
plt.ylabel('Residual Value')
plt.grid(True)
plt.show()


In [ ]:
start_index = len(X_train)
end_index = start_index + n_step - 1

# Generate the forecast with confidence intervals
forecast_sarimax = results_sarimax.predict(start=start_index, end=end_index, exog=X_test_scaled)
forecast_sarimax = np.expm1(forecast_sarimax.reshape(-1,1))
#forecast_sarimax  = scaler_y.inverse_transform(forecast_sarimax.reshape(-1,1))

# Optional: Plot the forecast
plt.figure(figsize=(12, 6))
plt.plot(df_completo.index[start_index:end_index+1], y_test, label='Observed')
plt.plot(df_completo.index[start_index:end_index+1], forecast_sarimax, color='red', label='Forecast')
plt.title('SARIMAX Forecast')
plt.xlabel('Time')
plt.ylabel('Media Value')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Calculate MSE
mse_sarimax = mean_squared_error(y_test, forecast_sarimax)
print(f"Mean Squared Error (MSE): {mse_sarimax:.2f}")

# Calculate RMSE
rmse_sarimax = np.sqrt(mse_sarimax)
print(f"Root Mean Squared Error (RMSE): {rmse_sarimax:.2f}")

# Calculate MAE
mae_sarimax = mean_absolute_error(y_test, forecast_sarimax)
print(f"Mean Absolute Error (MAE): {mae_sarimax:.2f}")

# Calculate MAPE
# Avoid division by zero if actual values are zero
mape_sarimax = np.mean(np.abs((y_test - forecast_sarimax) / y_test)) * 100
print(f"Mean Absolute Percentage Error (MAPE): {mape_sarimax:.2f}%")

# day

In [ ]:
df_completo = pd.read_csv('/Users/Administrador/Documents/umidade_solo/df_frei_orestes_m.csv')
df_completo['datahora'] = pd.to_datetime(df_completo['datahora'])

In [ ]:
from darts.dataprocessing.transformers import Scaler
series = TimeSeries.from_dataframe(df_completo,
                                   time_col="datahora",
                                   value_cols="0 - 50 cm")
cov = TimeSeries.from_dataframe(df_completo,
                                time_col="datahora",
                                value_cols="precipitacao_14_dias")

# Split the data into train and test sets
train, remainder = series.split_before(0.7)#0.765
val, test = remainder.split_before(0.5)
cov_train, cov_remainder = cov.split_before(0.7)
cov_val, cov_test = cov_remainder.split_before(0.5)

scaler_yy = Scaler()
scaler_xx = Scaler()

train_scaled = scaler_yy.fit_transform(train)
val_scaled = scaler_yy.transform(val)
test_scaled = scaler_yy.transform(test)

cov_scaled = scaler_xx.fit_transform(cov)
cov_train_scaled = scaler_xx.transform(cov_train)
cov_val_scaled = scaler_xx.transform(cov_val)
cov_test_scaled = scaler_xx.transform(cov_test)

print("Shapes:")
print(train_scaled.shape, test_scaled.shape)
print(cov_train_scaled.shape, cov_test_scaled.shape)

In [ ]:
from pytorch_lightning.callbacks import Callback

class LossRecorder(Callback):
    def __init__(self):
        self.train_loss_history = []
        self.val_loss_history = []

    def on_train_epoch_end(self, trainer, pl_module):
        self.train_loss_history.append(trainer.callback_metrics["train_loss"].item())
        self.val_loss_history.append(trainer.callback_metrics["val_loss"].item())
loss_recorder = LossRecorder()

In [ ]:
model_lstm = BlockRNNModel(
    input_chunk_length=8,#30
    output_chunk_length=1,#1
    model="LSTM",
    hidden_dim=50,# 100 e 150cm -> 30, 50cm -> 20
    n_rnn_layers=2,
    dropout=0.1,#0
    batch_size=64,#16
    n_epochs=200,#100
    optimizer_kwargs={"lr": 1e-3},
    model_name="Air_RNN",
    log_tensorboard=True,
    random_state=42,
    force_reset=True,
    use_static_covariates=True,
    #add_encoders=add_encoders,
    pl_trainer_kwargs={"callbacks": [loss_recorder]},
    save_checkpoints=True
)

In [ ]:
start = time.time()
model_lstm.fit(
    train_scaled,
    #past_covariates=cov_train_scaled,
    val_series=val_scaled,
    #val_past_covariates=cov_val_scaled,
    #val_future_covariates=cov_scaled[['sin_ano', 'cos_ano']],
    #future_covariates=cov_scaled['sin_ano'],
    verbose=True,
)

end = time.time()
tempo_lstm = end - start

In [ ]:
forecast_lstm = model_lstm.predict(n=24)#, past_covariates=cov_scaled)#, future_covariates=cov_scaled['sin_ano'])
forecast_lstm = scaler_yy.inverse_transform(forecast_lstm)
plt.figure(figsize=(8, 5))
remainder.plot(label="test real")
forecast_lstm.plot(label="forecast")
plt.title(f"MAPE: {mape(forecast_lstm, test):.2f}%")
plt.legend()
plt.show()

In [ ]:
plt.plot(loss_recorder.train_loss_history, label='Train Loss')
plt.plot(loss_recorder.val_loss_history, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
'''historical_forecasts_train = model_lstm.historical_forecasts(
    series=train_scaled,
    past_covariates=cov_scaled,
    start=train_scaled.time_index[model_lstm.input_chunk_length],
    forecast_horizon=1,
    stride=1,
    retrain=False,
    verbose=True,
    last_points_only=False,
)
datas = [ts.time_index[0] for ts in historical_forecasts_train]
valores = [ts.values().item() for ts in historical_forecasts_train]
df_forecasts = pd.DataFrame({
    "datahora": datas,
    "forecast": valores
})
historical_forecasts_train = TimeSeries.from_dataframe(df_forecasts,
                                time_col="datahora",
                                value_cols="forecast")'''

In [ ]:
# Calcular métricas
mse_lstm = mse(test[:n_step], forecast_lstm[len(cov_test):])
rmse_lstm = rmse(test[:n_step], forecast_lstm[len(cov_test):])
mae_lstm = mae(test[:n_step], forecast_lstm[len(cov_test):])
mape_lstm = mape(test[:n_step], forecast_lstm[len(cov_test):])

print(f"MSE: {mse_lstm:.4f}")
print(f"RMSE: {rmse_lstm:.4f}")
print(f"MAE: {mae_lstm:.4f}")
print(f"MAPE: {mape_lstm:.2f}%")

# test

In [ ]:
path = '/Users/Administrador/Downloads/Consumo_horario_2024_05.csv'
df = pd.read_csv(path, sep=';')#, nrows=100)

In [ ]:
df.columns

In [ ]:
rj = df[df["Cidade"] == 'RIO DE JANEIRO']

In [ ]:
#rj[rj["Ramo de Atividade"] == 'COMÉRCIO']
len(rj)

In [ ]:
#'Consumo de energia ajustado de uma parcela de carga - MWh (RC c,j)',

In [ ]:
rj['Ramo de Atividade'].unique()

In [ ]:
#df.to_csv('/Users/Administrador/Downloads/Consumo.csv', index=False)
